# Phy-World: reproducir «Escalas del futuro» y experimentar

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QU01/Phy-World/blob/ccr-3d0beba9-kg1125/notebooks/reproducir_reporte.ipynb)

Este notebook clona **QUASAR Phy-World** desde GitHub, instala sus datos y reproduce los dos informes:

1. **El árbol de los futuros** (`reports/arbol_decision`): árbol de decisión de 5 políticas × geología sobre 500 mundos calibrados con la historia, más un árbol CART aprendido. Sale como página HTML interactiva.
2. **Escalas del futuro** (`reports/escenarios`): cada política a 7 intensidades, la anatomía del valle de 2040–2060, la geopolítica y sociedad en 180 países (capa L1) y el PDF completo.
3. **Sociedad y límites biofísicos** y **El modelo acoplado** (`reports/acople`): la evidencia histórica y el modelo World3 ↔ 180 países que la implementa.

Al final hay una sección **Experimenta** para probar tus propias políticas, escalas, supuestos y escenarios geopolíticos.

| Paso | Tiempo aproximado (Colab, CPU) |
|---|---|
| Instalación | 2–4 min |
| Datos (Atlas, población WPP, energía) | 3–6 min |
| Árbol de decisión | 3–5 min |
| Estudio de escalas | 3–5 min |
| Geopolítica (L1, 5 escenarios × 64 mundos) | 4–10 min (`RAPIDO = True`: ≈ 1 min) |
| PDF | < 1 min |
| Sociedad y modelo acoplado (2 PDF) | ≈ 5 min |

> **Cómo leer los resultados.** Son *modos de comportamiento* de los modelos, no pronósticos. El núcleo es World3-03 (*Limits to Growth*, 2004). La capa de 180 países está validada solo a 1–10 años. Los límites de cada parte están en el propio PDF.

## 0. Parámetros
Cambia `BRANCH` a `"main"` cuando el PR #1 esté fusionado. `RAPIDO = True` reduce la corrida geopolítica a 8 mundos por escenario para probar en poco tiempo.

In [ ]:
REPO = "https://github.com/QU01/Phy-World"
BRANCH = "ccr-3d0beba9-kg1125"   # la rama con los informes; "main" cuando el PR #1 esté fusionado
WORKDIR = "Phy-World"           # carpeta donde se clona (si el notebook no está ya dentro del repo)
RAPIDO = False                  # True: geopolítica con 8 mundos por escenario en lugar de 64

## 1. Clonar Phy-World e instalarlo
Si abres el notebook desde un clon local del repositorio, usa ese clon; si no, clona `REPO` en `WORKDIR`.

In [ ]:
import os, subprocess, sys
from pathlib import Path

here = Path.cwd()
if (here / "world4").is_dir():
    ROOT = here
elif (here.parent / "world4").is_dir():          # abierto desde notebooks/
    ROOT = here.parent
else:
    if not Path(WORKDIR).exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, WORKDIR], check=True)
    ROOT = Path(WORKDIR).resolve()
os.chdir(ROOT)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"Phy-World en {ROOT} (commit {commit})")

In [ ]:
# Phy-World fija versiones exactas (numpy, scipy, pandas, statsmodels) y trae phycore desde GitHub.
# [wpp] añade el lector de los datos de población de la ONU; el resto es para los informes.
%pip install -q -e ".[wpp]" scikit-learn matplotlib reportlab pillow pypdf jinja2

In [ ]:
# Si pip cambió una librería que ya estaba cargada (pasa en Colab con numpy), hay que reiniciar el kernel.
import importlib.metadata as md_
stale = [m for m in ("numpy", "scipy", "pandas") if m in sys.modules
         and getattr(sys.modules[m], "__version__", None) != md_.version(m)]
if stale:
    print("Se actualizaron", stale, "-> reinicia el kernel y vuelve a ejecutar desde el paso 0.")
    if "google.colab" in sys.modules:
        os.kill(os.getpid(), 9)                  # Colab reinicia la sesión solo
else:
    import numpy, world4
    print("listo: numpy", numpy.__version__, "· Phy-World MODEL_REVISION", world4.MODEL_REVISION)

## 2. Datos
Phy-World no incluye los datos (varias fuentes no se pueden redistribuir). Cada instalador descarga los ficheros y comprueba su SHA-256 contra un manifiesto.

**Nota sobre el Atlas.** El manifiesto del Atlas se generó en Windows, con finales de línea CRLF. En Linux o Colab los mismos datos llegan con LF y la verificación falla. La función de abajo convierte a CRLF **solo** los ficheros cuyo hash coincide así; si algún fichero no coincide ni con CRLF, se detiene.

Los datos de clima (FaIR, Zenodo) y alimentos (FAOSTAT) son opcionales: los informes no los usan. Puedes instalarlos con `world4_cli.py --install-climate --install-food` si tu red llega a esos servidores.

In [ ]:
import hashlib, json, warnings
warnings.simplefilter("ignore")
from world4.data import atlas, wpp, energy

def ajustar_crlf_atlas():
    man = atlas.manifest()["files"]
    base = atlas.root()
    convertidos, malos = 0, []
    for rel, meta in man.items():
        p = base / rel
        if not p.is_file():
            continue
        b = p.read_bytes()
        if hashlib.sha256(b).hexdigest() == meta["sha256"]:
            continue
        c = b.replace(b"\r\n", b"\n").replace(b"\n", b"\r\n")
        if hashlib.sha256(c).hexdigest() == meta["sha256"]:
            p.write_bytes(c); convertidos += 1
        else:
            malos.append(rel)
    if malos:
        raise RuntimeError(f"ficheros del Atlas que no coinciden ni con CRLF: {malos}")
    return convertidos

if not atlas.verify()["ok"]:
    print("instalando el Atlas (git clone en el commit fijado)...")
    try:
        atlas.install()
    except Exception as e:                       # la verificación final la hace verify()
        print("aviso:", e)
    print("ficheros convertidos a CRLF:", ajustar_crlf_atlas())
print("Atlas:", "OK" if atlas.verify()["ok"] else atlas.verify())

if not wpp.available():
    print("instalando UN WPP 2024 (población por edad y sexo)...")
    wpp.install()
print("WPP 2024:", "OK" if wpp.available() else "FALTA")

if not energy.available():
    print("instalando energía (OWID)...")
    energy.install()
print("Energía:", "OK" if energy.available() else "FALTA")

## 3. Comprobación rápida: el núcleo World3-03
El escenario 1 de *Limits to Growth* («business as usual»), con la población observada de la ONU encima. Phy-World verifica este modelo contra Vensim con un error máximo de 6,9·10⁻⁶.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from world4.core import l0

r = l0.run(scenario=1)
t = r.time
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for ax, (k, titulo, esc) in zip(axes, [("pop", "Población (miles de millones)", 1e-9), ("iopc", "Producción industrial p. c.", 1),
                                      ("fpc", "Alimento p. c.", 1), ("ppolx", "Contaminación (1970 = 1)", 1)]):
    ax.plot(t, r[k][:, 0] * esc); ax.set_title(titulo); ax.set_xlim(1900, 2100); ax.axvline(2026, color="gray", lw=0.8)
d = np.load(ROOT / "data_cache" / "wpp2024" / "wpp_age.npz")
axes[0].plot(d["years"], (d["popM"].sum(2) + d["popF"].sum(2)).sum(0) * 1e3 / 1e9, "k.", ms=3, label="ONU (WPP 2024)")
axes[0].legend(); plt.tight_layout(); plt.show()

## 4. Informe 1: el árbol de los futuros
`proyeccion.py` arma el ensemble (6.000 candidatos → 500 mundos que reproducen la población de 2000 y 2020), corre las 64 hojas del árbol, el CART y el barrido del coste de esperar. `construir.py` genera la página interactiva.

In [ ]:
sys.path[:0] = [str(ROOT / "reports" / "arbol_decision"), str(ROOT / "reports" / "escenarios")]
import proyeccion
proyeccion.main()                     # escribe reports/arbol_decision/data.json
subprocess.run([sys.executable, str(ROOT / "reports" / "arbol_decision" / "construir.py")], check=True)

In [ ]:
import pandas as pd
arbol = json.loads((ROOT / "reports/arbol_decision/data.json").read_text())
pol = pd.DataFrame(arbol["policies"])
pol["palancas"] = pol.apply(lambda p: ", ".join(n for n, k in (("limpia", "clean"), ("agro", "agro"), ("demografía", "demo"),
                                                              ("suficiencia", "suff")) if p[k]) or "ninguna", axis=1)
for j, nombre in enumerate(arbol["destinies"]):
    pol[nombre] = pol["p_destiny"].str[j]
pol[["year", "palancas", "value"] + arbol["destinies"]].rename(columns={"year": "año", "value": "bienestar"}).head(10).style.format(
    {"bienestar": "{:.3f}", **{d: "{:.0%}" for d in arbol["destinies"]}})

In [ ]:
# La página interactiva, aquí mismo (D3 se carga de un CDN). También está en reports/arbol_decision/informe.html
import html
from IPython.display import HTML
pagina = (ROOT / "reports/arbol_decision/informe.html").read_text()
HTML(f'<iframe srcdoc="{html.escape(pagina)}" style="width:100%;height:900px;border:0"></iframe>')

## 5. Informe 2: escalas del futuro
### 5.1 Estudio por escalas (núcleo World3)
Dosis-respuesta de cada palanca (7 niveles, sola y en paquete, dos geologías), intensidad × año de arranque, geología continua, interacciones, nueve escenarios compuestos y la anatomía del valle. Escribe `reports/escenarios/estudio.json`.

In [ ]:
import estudio
estudio.main()
E = json.loads((ROOT / "reports/escenarios/estudio.json").read_text())
DEST = E["destinies"]

def tabla_dosis(palanca, contexto="paquete", geo="abund"):
    filas = []
    for r in E["dose"][palanca][contexto][geo]:
        fila = {"nivel": r["level"], **{d: r["p_destiny"][j] for j, d in enumerate(DEST)},
                "bienestar": r["hwi_mean"]["mean"], "EV mínima": r["le_min"]["q"][1], "EV 2100": r["le_2100"]["q"][1],
                "huella 2100": r["hef_2100"]["q"][1]}
        filas.append(fila)
    return pd.DataFrame(filas).style.format({**{d: "{:.0%}" for d in DEST}, "bienestar": "{:.3f}", "EV mínima": "{:.0f}",
                                             "EV 2100": "{:.0f}", "huella 2100": "{:.2f}"})

tabla_dosis("agro")      # prueba: "poll", "res", "agro", "soil", "fam", "suff"; contexto "solo" o "paquete"

In [ ]:
# La anatomía del valle: cuándo cae la esperanza de vida y qué factor la hunde
pd.DataFrame([{"caso": c["name"], "valle (mediana)": c["valley_year"][1], "EV 2026": c["le_2026"], "EV valle": c["le_valley"],
               "EV 2100": c["le_2100"], **{f"por {k}": v for k, v in c["share"].items()}} for c in E["valley"]["cases"]]).style.format(
    {"valle (mediana)": "{:.0f}", "EV 2026": "{:.0f}", "EV valle": "{:.0f}", "EV 2100": "{:.0f}",
     **{f"por {k}": "{:.0%}" for k in ("comida", "salud", "contaminacion", "hacinamiento")}})

### 5.2 Geopolítica y sociedad (capa L1, 180 países)
Corre el Atlas acoplado desde los datos de 2019 hasta 2100 en cinco escenarios, en paralelo. Escribe `reports/escenarios/geopolitica.json`.

In [ ]:
if RAPIDO:
    os.environ["W4_CHUNKS"], os.environ["W4_PER_CHUNK"] = "2", "4"
os.environ["PYTHONPATH"] = os.pathsep.join([str(ROOT), str(ROOT / "reports" / "escenarios")])   # para los procesos hijos
import geopolitica
geopolitica.CHUNKS = int(os.environ.get("W4_CHUNKS", 4)); geopolitica.PER_CHUNK = int(os.environ.get("W4_PER_CHUNK", 16))
geopolitica.main()
GP = json.loads((ROOT / "reports/escenarios/geopolitica.json").read_text())
ix = [GP["years"].index(y) for y in (2030, 2050, 2075, 2100)]
pd.DataFrame({(GP["scenarios"][k]["name"], m): [GP["world"][k][m][1][i] for i in ix]
              for k in GP["world"] for m in ("psi_index", "conflicts", "demo_pop")},
             index=[2030, 2050, 2075, 2100]).round(2)

### 5.3 El PDF
Las fuentes (Instrument Serif e IBM Plex, licencia OFL) se descargan de GitHub la primera vez.

In [ ]:
import importlib, informe_pdf
importlib.reload(informe_pdf)          # vuelve a leer estudio.json y geopolitica.json recién generados
informe_pdf.build()
pdf = ROOT / "reports/escenarios/escalas_del_futuro.pdf"
print(pdf, f"{pdf.stat().st_size / 1e6:.1f} MB")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(pdf))
%matplotlib inline

### 5.4 Sociedad y límites biofísicos: la investigación y el modelo acoplado
Dos informes más en `reports/acople`:
- `investigacion.py` + `informe_dinamicas.py`: qué dicen 70 años de datos de 180 países sobre cómo se conectan el ingreso, la comida, los recursos y la contaminación con el conflicto, los golpes, la democracia y las deudas (`dinamicas_sociedad_biofisica.pdf`).
- `calibrar.py` + `simular.py` + `informe_acoplado.py`: el **modelo acoplado** World3 ↔ 180 países (`world4/core/social.py`), calibrado y validado con esos datos, corrido sobre los 500 mundos (`modelo_acoplado.pdf`). `calibrar.py` descarga una vez las proyecciones de población de la ONU (≈ 34 MB, GitHub).

Unos 5 minutos en total.

In [ ]:
for script in ("investigacion.py", "informe_dinamicas.py", "calibrar.py", "simular.py", "informe_acoplado.py"):
    print("→", script)
    subprocess.run([sys.executable, str(ROOT / "reports" / "acople" / script)], check=True, cwd=ROOT,
                   env={**os.environ, "PYTHONPATH": str(ROOT)})
print("PDF:", ROOT / "reports/acople/modelo_acoplado.pdf")

## 6. Experimenta
Todo lo de abajo usa el **mismo ensemble de 500 mundos** que los informes, así que tus resultados se comparan directamente con los del PDF.

### 6.1 Tu propia política en World3
Una política es un diccionario con seis palancas:

| clave | qué es | rango usado en el informe | referencia 1× |
|---|---|---|---|
| `poll` | mejora anual de la tecnología de control de la contaminación | 0–0,08 | 0,04 |
| `res` | mejora anual de la eficiencia en el uso de recursos | 0–0,08 | 0,04 |
| `agro` | mejora anual del rendimiento agrícola | 0–0,08 | 0,04 |
| `soil` | protección de suelos (0 o 1) | 0/1 | 1 |
| `fam` | hijos deseados por familia (0 = sin política) | 1,5–3 | 2 |
| `suff` | producción industrial per cápita deseada (0 = sin política) | 250–500 | 350 |

`geo` multiplica el recurso no renovable (1 = World3; mayor que 1 incluye mejor tecnología de extracción) y `year` es el año de arranque.

In [ ]:
%matplotlib inline
# (el paso del PDF deja matplotlib sin pantalla; esto vuelve a mostrar las gráficas aquí)
from proyeccion import sample_uncertain, constants_for, simulate, tables_for, observed_population, SEED, NEVER

def ensemble(n=500):
    rng = np.random.default_rng(SEED)
    obs = observed_population()
    cand = sample_uncertain(6000, rng)
    c = constants_for(cand, res_abundant=np.zeros(6000), year=NEVER, clean=False, agro=False, demo=False, suff=False)
    y, s = simulate(cand, tables_for(False, False), c)
    p00, p20 = s["pop"][np.searchsorted(y, 2000)] * 1e-9, s["pop"][np.searchsorted(y, 2020)] * 1e-9
    ok = (abs(p00 / obs[2000] - 1) < 0.06) & (abs(p20 / obs[2020] - 1) < 0.06)
    idx = np.flatnonzero(ok)[:n]
    return {k: v[idx] for k, v in cand.items()}

U = ensemble()
print(len(U["nri_mult"]), "mundos")

def probar(pol, geo=2.0, year=2027, ver=("pop", "le", "hef", "ppolx", "fpc", "iopc"), titulo="Mi política"):
    pol = {**estudio.OFF, **pol}
    y, s = estudio.run(U, pol, geo=geo, year=float(year))
    yb, sb = estudio.run(U, estudio.OFF, geo=geo)
    o = estudio.outcomes(y, s)
    p = {d: float(np.mean(o["destiny"] == j)) for j, d in enumerate(DEST)}
    print(titulo, "·", " · ".join(f"{d}: {v:.0%}" for d, v in p.items()))
    print(f"bienestar medio {o['hwi_mean'].mean():.3f} · EV mínima {np.median(o['le_min']):.0f} · EV 2100 "
          f"{np.median(o['le_2100']):.0f} · población 2100 {np.median(o['pop_2100']):.1f} mm · huella 2100 {np.median(o['hef_2100']):.2f}")
    fig, axes = plt.subplots(1, len(ver), figsize=(3 * len(ver), 2.6))
    for ax, v in zip(axes, ver):
        esc = proyeccion.VARS[v][2]
        q = np.percentile(s[v] * esc, [10, 50, 90], axis=1)
        ax.fill_between(y, q[0], q[2], alpha=0.2, color="tab:green")
        ax.plot(y, q[1], color="tab:green", label=titulo)
        ax.plot(yb, np.median(sb[v] * esc, 1), color="gray", label="inercia")
        ax.set_title(proyeccion.VARS[v][0], fontsize=9); ax.set_xlim(1950, 2100); ax.axvline(year, ls=":", color="tab:green")
    axes[0].legend(fontsize=7); plt.tight_layout(); plt.show()
    return p

probar({"poll": 0.05, "res": 0.03, "agro": 0.05, "soil": 1, "fam": 1.8, "suff": 320}, geo=2.0, year=2027)

In [ ]:
# El mismo paquete arrancando en 2040 y con los recursos de World3 (geo = 1)
probar({"poll": 0.05, "res": 0.03, "agro": 0.05, "soil": 1, "fam": 1.8, "suff": 320}, geo=1.0, year=2040, titulo="Tarde y con poca geología")

### 6.2 Barre una palanca a tus propios niveles

In [ ]:
def barrer(palanca, niveles, base=None, geo=2.0, year=2027):
    base = {**estudio.OFF, **(base if base is not None else estudio.REF)}
    filas = []
    for v in niveles:
        y, s = estudio.run(U, {**base, palanca: v}, geo=geo, year=float(year))
        o = estudio.outcomes(y, s)
        filas.append({"nivel": v, **{d: float(np.mean(o["destiny"] == j)) for j, d in enumerate(DEST)},
                      "bienestar": o["hwi_mean"].mean(), "EV mínima": np.median(o["le_min"])})
    df = pd.DataFrame(filas).set_index("nivel")
    df[DEST].plot.barh(stacked=True, figsize=(8, 0.35 * len(niveles) + 1), width=0.8,
                       color=["#b3302b", "#eb8a7f", "#cbc5b8", "#78aee8", "#1c5cab"])
    plt.legend(fontsize=7, bbox_to_anchor=(1, 1)); plt.xlabel("probabilidad"); plt.tight_layout(); plt.show()
    return df.style.format({**{d: "{:.0%}" for d in DEST}, "bienestar": "{:.3f}", "EV mínima": "{:.0f}"})

barrer("poll", [0, 0.015, 0.03, 0.045, 0.06, 0.1])     # control de la contaminación, con el resto del paquete en 1×

### 6.3 Cambia los supuestos del modelo
`estudio.run` acepta cualquier política; para tocar el modelo en sí, `world4.core.l0.run` acepta constantes y tablas de World3-03 (`l0.CONSTANTS`, `l0.TABLES`). Por ejemplo: ¿qué pasa si la contaminación tarda 40 años en transmitirse en lugar de 15–30?

In [ ]:
U_lento = {**U, "pptd": np.full_like(U["pptd"], 40.0)}
U_orig, U = U, U_lento
probar(estudio.REF, geo=2.0, titulo="Paquete 1×, transmisión de 40 años")
U = U_orig

### 6.4 Tu propio escenario geopolítico (capa L1)
Un escenario del L1 combina:
- `calib`: perillas por año (`intensity` y `nonfossil` de energía, `tfr` y `e0` de población);
- `conduct=True`: la conducta que «les convendría» a los países según el problema inverso del Atlas;
- `g0_after`: la deriva anual de la frontera tecnológica desde 2020 (≈ +0,003 en los datos; negativa = estancamiento).

El ejemplo combina transición fuerte y autonomía con una frontera tecnológica que deja de avanzar (`g0_after = 0`): verás que la transición no alcanza para compensar el frenazo y la presión política sube. Cambia `g0_after` a `None` para dejar la productividad como en los datos. Con 8 mundos tarda alrededor de un minuto (aquí, unos segundos).

In [ ]:
geopolitica.SCENARIOS["mio"] = {"name": "Mi escenario", "calib": {"intensity": -0.02, "nonfossil": 0.08, "tfr": -0.003},
                                "conduct": True, "g0_after": 0.0}
geopolitica.PER_CHUNK = 8
_, mundo, region, pais, iso, ridx = geopolitica._chunk(("mio", 1000))
psi = 100 * np.exp(mundo["lpsi"] - mundo["lpsi"][:, :1])
anios = np.array(GP["years"])
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, (k, titulo) in zip(axes, [("psi", "Presión política (2019 = 100)"), ("conflicts", "Países en conflicto civil"),
                                   ("demo_pop", "Población en democracia")]):
    mio = np.median(psi if k == "psi" else mundo[k], 0)
    ref = GP["world"]["inercia"]["psi_index" if k == "psi" else k][1]
    ax.plot(anios, ref, color="gray", label="Inercia (64 mundos)")
    ax.plot(anios, mio, color="tab:purple", label="Mi escenario (8 mundos)")
    ax.set_title(titulo)
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

### 6.5 World3 y la sociedad, retroalimentándose
`world4.core.social` acopla World3 con 180 países (ver `modelo_acoplado.pdf`). El gancho `social.Coupler` se le pasa a `estudio.run`, así que funciona con cualquier política de 6.1. Se corre dos veces con los mismos números aleatorios: **sólo ida** (la sociedad sigue a World3 sin tocarlo) y **con vuelta** (los eventos destruyen capital y restan capacidad de política). Prueba otras políticas, `geo=1.0` (colapso) o las pruebas de estrés `impulse_scale=2`, `hazard_scale=1.5`.

In [ ]:
from world4.core import social

mi_pol, mi_geo = estudio.REF, 2.0          # cambia aquí
estres = {}                               # p. ej. {"impulse_scale": 2.0, "hazard_scale": 1.5}
corridas = {}
for modo, fb in (("sólo ida", False), ("con vuelta", True)):
    gancho = social.Coupler(len(U["nri_mult"]), seed=1, feedback=fb, capacity=fb, **estres)
    anios, s = estudio.run(U, mi_pol, geo=mi_geo, hook=gancho)
    corridas[modo] = (anios, s, gancho.result())

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
soc = corridas["con vuelta"][2]
for ax, (k, titulo) in zip(axes[:3], [("dem_breakdown", "Quiebres democráticos / año"), ("sovdebt_onset", "Crisis de deuda / año"),
                                      ("conflict_onset", "Inicios de conflicto / año")]):
    ax.plot(soc.years, soc[k].mean(1), color="tab:red"); ax.set_title(titulo)
for modo, color in (("sólo ida", "gray"), ("con vuelta", "tab:red")):
    anios, s, _ = corridas[modo]
    axes[3].plot(anios, np.median(s["le"], 1), color=color, label=modo)
axes[3].set_xlim(2000, 2100); axes[3].set_title("Esperanza de vida (mediana)"); axes[3].legend(fontsize=8)
plt.tight_layout(); plt.show()
ida, vuelta = (estudio.outcomes(*corridas[m][:2]) for m in ("sólo ida", "con vuelta"))
print("Destinos sólo ida :", np.bincount(ida["destiny"], minlength=5) / len(ida["destiny"]))
print("Destinos con vuelta:", np.bincount(vuelta["destiny"], minlength=5) / len(vuelta["destiny"]))
print(f"PIB mundial destruido por los eventos a 2100 (mediana): {1 - np.median(soc['gdp_index'][-1]):.1%}")

## 7. Dónde seguir
- **Código de los informes:** `reports/arbol_decision/proyeccion.py`, `reports/escenarios/estudio.py`, `geopolitica.py` e `informe_pdf.py`. Cambia niveles, escenarios o umbrales de destino y vuelve a ejecutar los pasos 4–5.
- **Modelo:** `world4/core/l0.py` (World3-03), `world4/core/social.py` (acople World3 ↔ sociedad, parámetros en `world4/data/social_params.json`), `world4/core/l1_atlas/` (Atlas), `world4/api.py` (superficie pública).
- **Límites:** el núcleo World3 no tiene Estados ni conflicto; el L1 está validado a 1–10 años; sin datos de clima y alimentos, la temperatura del L1 no responde a la transición. Están detallados en el PDF.